## O2 Structural Feature Comparison

This notebook compares O2 overpotential prediction models trained with and without structural features across multiple iterations.

Workflow:
1. Load paired datasets with and without structural features.
2. Train and evaluate TabPFN models using compound-level splitting.
3. Perform five-fold grouped cross-validation.
4. Compare R², MAE, and RMSE on the cross-validation and test sets.
5. Save the final comparison results to a CSV file.

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from tabpfn import TabPFNRegressor

In [2]:
# ============================================================
# 2. Configuration
# ============================================================

GROUP_COL = "compound"

TEST_SIZE = 0.20
N_SPLITS = 5
RANDOM_STATE = 42


file_pairs = {
    0: {
        "Without_structure":
            "data/compound_0_O2_nostructural.csv",
        "With_structure":
            "data/compound_0_O2.csv"
    },
    1: {
        "Without_structure":
            "data/compound_1_O2_nostructural.csv",
        "With_structure":
            "data/compound_1_O2.csv"
    },
    2: {
        "Without_structure":
            "data/compound_2_O2_nostructural.csv",
        "With_structure":
            "data/compound_2_O2.csv"
    },
    3: {
        "Without_structure":
            "data/compound_3_O2_nostructural.csv",
        "With_structure":
            "data/compound_3_O2.csv"
    },
    4: {
        "Without_structure":
            "data/compound_4_O2_nostructural.csv",
        "With_structure":
            "data/compound_4_O2.csv"
    },
    5: {
        "Without_structure":
            "data/compound_5_O2_nostructural.csv",
        "With_structure":
            "data/compound_5_O2.csv"
    },
    6: {
        "Without_structure":
            "data/compound_6_O2_nostructural.csv",
        "With_structure":
            "data/compound_6_O2.csv"
    }
}

os.makedirs(
    "results/structure_comparison",
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Define model and evaluation metrics
# ============================================================

def create_model():

    model = TabPFNRegressor(
        device="cuda",
        model_path=(
            "tabpfn_models/"
            "tabpfn-v3-regressor-v3_default.ckpt"
        ),
        n_estimators=64,
        random_state=RANDOM_STATE
    )

    return model


def calculate_metrics(
    y_true,
    y_pred
):

    r2 = r2_score(
        y_true,
        y_pred
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    return {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }

In [4]:
# ============================================================
# 4. Train and evaluate all iterations
# ============================================================

all_cv_results = []
all_test_results = []


for iteration, paths in file_pairs.items():

    print("\n" + "=" * 70)
    print(f"Iteration {iteration}")
    print("=" * 70)

    # --------------------------------------------------------
    # Evaluate the two feature settings independently
    # --------------------------------------------------------

    for feature_setting, data_path in paths.items():

        print("\n" + "-" * 70)
        print(feature_setting)
        print("-" * 70)

        # ----------------------------------------------------
        # Load data
        # ----------------------------------------------------

        data = pd.read_csv(
            data_path
        )

        data = data.loc[
            :,
            ~data.columns.str.startswith("Unnamed")
        ].copy()

        data = data.reset_index(
            drop=True
        )

        # The last column is the target
        target_col = data.columns[-1]

        # ----------------------------------------------------
        # Prepare features, target, and groups
        # ----------------------------------------------------

        groups = (
            data[GROUP_COL]
            .astype(str)
            .reset_index(drop=True)
        )

        y = (
            data[target_col]
            .astype(float)
            .to_numpy(dtype=np.float32)
        )

        X = (
            data
            .drop(
                columns=[
                    GROUP_COL,
                    target_col
                ]
            )
            .apply(
                pd.to_numeric,
                errors="raise"
            )
            .astype(np.float32)
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Independent group train-test split
        # ----------------------------------------------------

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE
        )

        train_idx, test_idx = next(
            splitter.split(
                X,
                y,
                groups=groups
            )
        )

        X_train = (
            X.iloc[train_idx]
            .reset_index(drop=True)
        )

        X_test = (
            X.iloc[test_idx]
            .reset_index(drop=True)
        )

        y_train = y[train_idx]
        y_test = y[test_idx]

        groups_train = (
            groups.iloc[train_idx]
            .reset_index(drop=True)
        )

        groups_test = (
            groups.iloc[test_idx]
            .reset_index(drop=True)
        )

        print(
            "Total samples:",
            len(X)
        )

        print(
            "Training samples:",
            len(X_train)
        )

        print(
            "Test samples:",
            len(X_test)
        )

        print(
            "Number of features:",
            X.shape[1]
        )

        print(
            "Training compounds:",
            groups_train.nunique()
        )

        print(
            "Test compounds:",
            groups_test.nunique()
        )

        # ----------------------------------------------------
        # Five-fold group cross-validation
        # ----------------------------------------------------

        cv_splitter = GroupKFold(
            n_splits=N_SPLITS
        )

        for fold, (
            cv_train_idx,
            cv_valid_idx
        ) in enumerate(
            cv_splitter.split(
                X_train,
                y_train,
                groups=groups_train
            ),
            start=1
        ):

            model = create_model()

            model.fit(
                X_train.iloc[cv_train_idx],
                y_train[cv_train_idx]
            )

            valid_pred = model.predict(
                X_train.iloc[cv_valid_idx]
            )

            cv_metrics = calculate_metrics(
                y_train[cv_valid_idx],
                valid_pred
            )

            all_cv_results.append({
                "Iteration": iteration,
                "Feature_setting": feature_setting,
                "Fold": fold,
                "N_train_fold": len(cv_train_idx),
                "N_valid_fold": len(cv_valid_idx),
                "R2": cv_metrics["R2"],
                "MAE": cv_metrics["MAE"],
                "RMSE": cv_metrics["RMSE"]
            })

            print(
                f"Fold {fold} | "
                f"R2 = {cv_metrics['R2']:.4f} | "
                f"MAE = {cv_metrics['MAE']:.4f} | "
                f"RMSE = {cv_metrics['RMSE']:.4f}"
            )

        # ----------------------------------------------------
        # Train final model
        # ----------------------------------------------------

        final_model = create_model()

        final_model.fit(
            X_train,
            y_train
        )

        # ----------------------------------------------------
        # Outer-test evaluation
        # ----------------------------------------------------

        test_pred = final_model.predict(
            X_test
        )

        test_metrics = calculate_metrics(
            y_test,
            test_pred
        )

        all_test_results.append({
            "Iteration": iteration,
            "Feature_setting": feature_setting,
            "N_features": X.shape[1],
            "N_total": len(X),
            "N_train": len(X_train),
            "N_test": len(X_test),
            "N_train_compounds": groups_train.nunique(),
            "N_test_compounds": groups_test.nunique(),
            "Test_R2": test_metrics["R2"],
            "Test_MAE": test_metrics["MAE"],
            "Test_RMSE": test_metrics["RMSE"]
        })

        print("\nOuter-test results:")

        print(
            f"{feature_setting:20s} | "
            f"Test R2 = {test_metrics['R2']:.4f} | "
            f"Test MAE = {test_metrics['MAE']:.4f} | "
            f"Test RMSE = {test_metrics['RMSE']:.4f}"
        )


Iteration 0

----------------------------------------------------------------------
Without_structure
----------------------------------------------------------------------
Total samples: 27
Training samples: 22
Test samples: 5
Number of features: 26
Training compounds: 20
Test compounds: 5
Fold 1 | R2 = -4.9442 | MAE = 0.4451 | RMSE = 0.5050
Fold 2 | R2 = -0.0114 | MAE = 0.2889 | RMSE = 0.3172
Fold 3 | R2 = -4.9248 | MAE = 0.5700 | RMSE = 0.6176
Fold 4 | R2 = -1.1698 | MAE = 0.2575 | RMSE = 0.2784
Fold 5 | R2 = 0.1107 | MAE = 0.2927 | RMSE = 0.3904

Outer-test results:
Without_structure    | Test R2 = 0.1585 | Test MAE = 0.1714 | Test RMSE = 0.2043

----------------------------------------------------------------------
With_structure
----------------------------------------------------------------------
Total samples: 27
Training samples: 22
Test samples: 5
Number of features: 34
Training compounds: 20
Test compounds: 5
Fold 1 | R2 = -5.3754 | MAE = 0.4579 | RMSE = 0.5230
Fold 2 | R2

In [5]:
# ============================================================
# 5. Convert results to DataFrames
# ============================================================

cv_results = pd.DataFrame(
    all_cv_results
)

test_results = pd.DataFrame(
    all_test_results
)

In [6]:
# ============================================================
# 6. Calculate cross-validation summary
# ============================================================

cv_summary = (
    cv_results
    .groupby(
        [
            "Iteration",
            "Feature_setting"
        ],
        as_index=False
    )
    .agg(
        CV_R2_mean=("R2", "mean"),
        CV_R2_std=("R2", "std"),
        CV_MAE_mean=("MAE", "mean"),
        CV_MAE_std=("MAE", "std"),
        CV_RMSE_mean=("RMSE", "mean"),
        CV_RMSE_std=("RMSE", "std")
    )
)

In [7]:
# ============================================================
# 7. Combine CV and test results
# ============================================================

summary_results = pd.merge(
    cv_summary,
    test_results,
    on=[
        "Iteration",
        "Feature_setting"
    ],
    how="left"
)

summary_results = summary_results[
    [
        "Iteration",
        "Feature_setting",
        "N_features",
        "N_total",
        "N_train",
        "N_test",
        "N_train_compounds",
        "N_test_compounds",
        "CV_R2_mean",
        "CV_R2_std",
        "CV_MAE_mean",
        "CV_MAE_std",
        "CV_RMSE_mean",
        "CV_RMSE_std",
        "Test_R2",
        "Test_MAE",
        "Test_RMSE"
    ]
]

In [8]:
# ============================================================
# 8. Display final results
# ============================================================

print("\n" + "=" * 100)
print("Final structure feature comparison")
print("=" * 100)

print(
    summary_results.to_string(
        index=False
    )
)


Final structure feature comparison
 Iteration   Feature_setting  N_features  N_total  N_train  N_test  N_train_compounds  N_test_compounds  CV_R2_mean  CV_R2_std  CV_MAE_mean  CV_MAE_std  CV_RMSE_mean  CV_RMSE_std   Test_R2  Test_MAE  Test_RMSE
         0    With_structure          34       27       22       5                 20                 5   -2.225376   2.870554     0.367860    0.149209      0.418859     0.162587  0.103625  0.172325   0.210890
         0 Without_structure          26       27       22       5                 20                 5   -2.187900   2.556614     0.370840    0.133057      0.421701     0.139394  0.158468  0.171423   0.204337
         1    With_structure          34      141      117      24                 35                 9   -0.771970   1.954841     0.123595    0.077675      0.240892     0.146495 -0.406092  0.141698   0.234524
         1 Without_structure          26      141      117      24                 35                 9   -0.803236   2.5344

In [9]:
# ============================================================
# 9. Save all results
# ============================================================

# cv_results.to_csv(
#     "results/structure_comparison/cv_fold_results.csv",
#     index=False
# )

# cv_summary.to_csv(
#     "results/structure_comparison/cv_summary.csv",
#     index=False
# )

# test_results.to_csv(
#     "results/structure_comparison/test_results.csv",
#     index=False
# )

summary_results.to_csv(
    "results/structure_comparison/O2final_comparison.csv",
    index=False
)

print("\nResults saved to:")
print("results/structure_comparison/")


Results saved to:
results/structure_comparison/
